In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import os

# Kaggle directories
INPUT_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
COMPETITIONS_DIR = INPUT_DIR / "competitions"

# Confirm that competition data is attached
if not COMPETITIONS_DIR.exists():
    raise FileNotFoundError(
        "The competitions folder was not found. "
        "Attach the RSNA competition data to this notebook."
    )

print("Attached competitions:")

competition_folders = [
    path for path in COMPETITIONS_DIR.iterdir()
    if path.is_dir()
]

for path in competition_folders:
    print(" -", path.name)

# Find the competition folder containing train.csv
train_candidates = list(
    COMPETITIONS_DIR.glob("*/train.csv")
)

if not train_candidates:
    raise FileNotFoundError(
        "Could not find train.csv inside the attached competition folders."
    )

# Select the RSNA competition directory
rsna_candidates = [
    path for path in train_candidates
    if "knee" in path.parent.name.lower()
    or "rsna" in path.parent.name.lower()
]

if rsna_candidates:
    train_path = rsna_candidates[0]
else:
    train_path = train_candidates[0]

DATA_DIR = train_path.parent

print("\nSelected competition directory:")
print(DATA_DIR)

print("\nTop-level competition files and folders:")

for path in sorted(DATA_DIR.iterdir()):
    item_type = "folder" if path.is_dir() else "file"
    print(f" - [{item_type}] {path.name}")

print("\nSetup complete.")
print("Read-only input directory:", DATA_DIR)
print("Writable output directory:", WORKING_DIR)

In [ ]:
# Load the competition CSV files

train = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

# Display dataset sizes
print("Dataset shapes:")
print(f"train.csv:             {train.shape}")
print(f"train_series.csv:      {train_series.shape}")
print(f"test.csv:              {test.shape}")
print(f"test_series.csv:       {test_series.shape}")
print(f"sample_submission.csv: {sample_submission.shape}")

# Check column names
print("\ntrain.csv columns:")
print(train.columns.tolist())

print("\ntrain_series.csv columns:")
print(train_series.columns.tolist())

# Preview the main tables
print("\nTraining studies:")
display(train.head())

print("\nTraining series:")
display(train_series.head())

print("\nSample submission:")
display(sample_submission.head())

In [ ]:
# Define the twelve prediction targets

LABELS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

# Count known and missing labels for every condition

label_summary = pd.DataFrame({
    "Known": train[LABELS].notna().sum(),
    "Missing": train[LABELS].isna().sum(),
    "Positive": train[LABELS].eq(1).sum(),
    "Negative": train[LABELS].eq(0).sum(),
})

label_summary["Positive Rate"] = (
    label_summary["Positive"]
    / label_summary["Known"].replace(0, np.nan)
)

label_summary["Coverage"] = (
    label_summary["Known"] / len(train)
)

display(
    label_summary.style.format({
        "Positive Rate": "{:.2%}",
        "Coverage": "{:.2%}",
    })
)

# Determine how completely each study is labelled

known_labels_per_study = train[LABELS].notna().sum(axis=1)

fully_labeled = (known_labels_per_study == len(LABELS)).sum()
partially_labeled = (
    (known_labels_per_study > 0)
    & (known_labels_per_study < len(LABELS))
).sum()
unlabeled = (known_labels_per_study == 0).sum()

print("Study-level label availability:")
print(f"Fully labelled studies:     {fully_labeled:,}")
print(f"Partially labelled studies: {partially_labeled:,}")
print(f"Unlabelled studies:         {unlabeled:,}")
print(f"Total studies:              {len(train):,}")

# Analyze how many MRI series belong to each study

series_per_study = (
    train_series.groupby("StudyInstanceUID")
    .size()
)

print("\nMRI series per study:")
print(series_per_study.describe())

print("\nAnatomical planes:")
display(
    train_series["Anatomical_Plane"]
    .value_counts(dropna=False)
    .to_frame("Series count")
)

print("\nSequence characteristics:")
display(
    train_series[
        ["Fluid_Sensitive", "Fat_Suppression"]
    ].value_counts(dropna=False)
    .to_frame("Series count")
)

In [ ]:
import pydicom
import matplotlib.pyplot as plt
import math

# Create labelled and unlabelled subsets
fully_labelled_mask = train[LABELS].notna().all(axis=1)

labelled_train = train[fully_labelled_mask].copy()
unlabelled_train = train[~fully_labelled_mask].copy()

print("Fully labelled studies:", len(labelled_train))
print("Unlabelled studies:", len(unlabelled_train))

# Select the first fully labelled study
selected_row = labelled_train.iloc[0]
study_id = selected_row["StudyInstanceUID"]

print("\nSelected study:")
print(study_id)

print("\nRadiology report:")
print(selected_row["Report"])

print("\nVerified labels:")
display(
    selected_row[LABELS]
    .astype(int)
    .to_frame("Value")
)

# Retrieve every series belonging to this study
study_series = train_series[
    train_series["StudyInstanceUID"] == study_id
].copy()

print("\nMRI series information:")
display(study_series)

In [ ]:
import math
import pydicom
import matplotlib.pyplot as plt
import numpy as np


def get_slice_number(path):
    """Read slice ordering information without loading pixel data."""
    metadata = pydicom.dcmread(
        path,
        stop_before_pixels=True,
        force=True
    )

    return float(getattr(metadata, "InstanceNumber", 0))


def prepare_for_display(dicom):
    """Convert a DICOM pixel array into a displayable image."""
    image = dicom.pixel_array.astype(np.float32)

    slope = float(getattr(dicom, "RescaleSlope", 1))
    intercept = float(getattr(dicom, "RescaleIntercept", 0))
    image = (image * slope) + intercept

    low, high = np.percentile(image, [1, 99])
    image = np.clip(image, low, high)

    if high > low:
        image = (image - low) / (high - low)

    if getattr(
        dicom,
        "PhotometricInterpretation",
        ""
    ) == "MONOCHROME1":
        image = 1.0 - image

    return image


series_images = []

for _, series_row in study_series.iterrows():
    series_id = series_row["SeriesInstanceUID"]

    series_directory = (
        DATA_DIR
        / "train_series"
        / study_id
        / series_id
    )

    dicom_files = list(series_directory.glob("*.dcm"))

    if not dicom_files:
        print("No images found for:", series_id)
        continue

    dicom_files = sorted(
        dicom_files,
        key=get_slice_number
    )

    # Load the middle slice
    middle_index = len(dicom_files) // 2
    middle_path = dicom_files[middle_index]

    dicom = pydicom.dcmread(
        middle_path,
        force=True
    )

    image = prepare_for_display(dicom)

    title = (
        f"{series_row['Anatomical_Plane']}\n"
        f"Fluid-sensitive: "
        f"{series_row['Fluid_Sensitive']} | "
        f"Fat suppression: "
        f"{series_row['Fat_Suppression']}\n"
        f"{len(dicom_files)} slices"
    )

    series_images.append((image, title))


# Create the image grid
number_of_images = len(series_images)
number_of_columns = 3
number_of_rows = math.ceil(
    number_of_images / number_of_columns
)

fig, axes = plt.subplots(
    number_of_rows,
    number_of_columns,
    figsize=(15, 5 * number_of_rows),
    squeeze=False
)

axes = axes.flatten()

for axis in axes:
    axis.axis("off")

for axis, (image, title) in zip(
    axes,
    series_images
):
    axis.imshow(image, cmap="gray")
    axis.set_title(title)
    axis.axis("off")

positive_labels = [
    label
    for label in LABELS
    if selected_row[label] == 1
]

plt.suptitle(
    "Study: "
    + study_id
    + "\nPositive labels: "
    + ", ".join(positive_labels),
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze the radiology reports

report_analysis = train[
    ["StudyInstanceUID", "Report"] + LABELS
].copy()

report_analysis["Report"] = (
    report_analysis["Report"]
    .fillna("")
    .astype(str)
    .str.strip()
)

report_analysis["ReportLength"] = (
    report_analysis["Report"].str.len()
)

report_analysis["WordCount"] = (
    report_analysis["Report"]
    .str.split()
    .str.len()
)

report_analysis["FullyLabelled"] = (
    report_analysis[LABELS]
    .notna()
    .all(axis=1)
)

nonempty_reports = report_analysis[
    report_analysis["ReportLength"] > 0
]

print("Report overview:")
print(f"Total studies:          {len(report_analysis):,}")
print(f"Nonempty reports:       {len(nonempty_reports):,}")
print(
    f"Missing/empty reports:  "
    f"{(report_analysis['ReportLength'] == 0).sum():,}"
)
print(
    f"Unique reports:         "
    f"{nonempty_reports['Report'].nunique():,}"
)
print(
    f"Duplicate reports:      "
    f"{nonempty_reports['Report'].duplicated().sum():,}"
)

print("\nReport character lengths:")
display(
    nonempty_reports["ReportLength"]
    .describe()
    .to_frame("Characters")
)

print("\nReport word counts:")
display(
    nonempty_reports["WordCount"]
    .describe()
    .to_frame("Words")
)

# Display representative reports without overwhelming the notebook
pd.set_option("display.max_colwidth", 700)

sample_reports = (
    nonempty_reports
    .sample(n=min(15, len(nonempty_reports)), random_state=42)
    .copy()
)

sample_reports["Report Preview"] = (
    sample_reports["Report"]
    .str.slice(0, 700)
)

display(
    sample_reports[
        [
            "StudyInstanceUID",
            "FullyLabelled",
            "ReportLength",
            "WordCount",
            "Report Preview",
        ]
    ]
)

In [ ]:
# Install the language detector
!pip install -q langdetect

import pandas as pd
import numpy as np

from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 42

# Reload train.csv if the variable no longer exists
if "train" not in globals():
    if "DATA_DIR" not in globals():
        raise RuntimeError(
            "Run cell 0 first: DATA_DIR is not defined."
        )
    train = pd.read_csv(DATA_DIR / "train.csv")

if "LABELS" not in globals():
    raise RuntimeError(
        "Run cell 2 first: LABELS is not defined."
    )

# Recreate the report-analysis table
report_analysis = train[
    ["StudyInstanceUID", "Report"] + LABELS
].copy()

report_analysis["Report"] = (
    report_analysis["Report"]
    .fillna("")
    .astype(str)
    .str.strip()
)

report_analysis["ReportLength"] = (
    report_analysis["Report"].str.len()
)

report_analysis["WordCount"] = (
    report_analysis["Report"]
    .str.split()
    .str.len()
)

report_analysis["FullyLabelled"] = (
    report_analysis[LABELS]
    .notna()
    .all(axis=1)
)


def detect_language(text):
    text = str(text).strip()

    if not text:
        return "unknown"

    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


# Detect report languages
report_analysis["Language"] = (
    report_analysis["Report"]
    .apply(detect_language)
)

LANGUAGE_NAMES = {
    "en": "English",
    "de": "German",
    "es": "Spanish",
    "tr": "Turkish",
    "hr": "Croatian",
    "bs": "Bosnian",
    "sr": "Serbian",
    "bg": "Bulgarian",
    "ru": "Russian",
    "nl": "Dutch",
    "el": "Greek",
    "fr": "French",
    "it": "Italian",
    "pt": "Portuguese",
    "pl": "Polish",
    "ro": "Romanian",
    "sl": "Slovenian",
    "mk": "Macedonian",
    "uk": "Ukrainian",
    "cs": "Czech",
    "sk": "Slovak",
    "hu": "Hungarian",
    "lt": "Lithuanian",
    "et": "Estonian",
    "ca": "Catalan",
    "unknown": "Unknown",
}

report_analysis["LanguageName"] = (
    report_analysis["Language"]
    .map(LANGUAGE_NAMES)
    .fillna(report_analysis["Language"])
)

# Create and display the language summary
language_summary = (
    report_analysis
    .groupby(["Language", "LanguageName"])
    .size()
    .reset_index(name="Reports")
    .sort_values("Reports", ascending=False)
)

language_summary["Percentage"] = (
    language_summary["Reports"]
    / len(report_analysis)
)

display(
    language_summary.style.format({
        "Percentage": "{:.2%}"
    })
)

print(
    "Language detection complete:",
    f"{len(report_analysis):,} reports processed."
)

In [ ]:
# ---------------------------------------------------------
# Verify prerequisites
# ---------------------------------------------------------

required_variables = [
    "language_summary",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

missing_columns = {
    "Language",
    "LanguageName",
} - set(report_analysis.columns)

if missing_columns:
    missing_variables.append(
        "report_analysis columns: "
        + ", ".join(sorted(missing_columns))
    )

if missing_variables:
    raise RuntimeError(
        "Run cell 6 first. Missing prerequisites: "
        + ", ".join(missing_variables)
    )


# Create the verified validation set
verified_reports = report_analysis[
    report_analysis[LABELS].notna().all(axis=1)
].copy()

print(f"Verified studies: {len(verified_reports):,}")

# Verified studies by language
verified_language_summary = (
    verified_reports
    .groupby(["Language", "LanguageName"])
    .size()
    .reset_index(name="Verified Reports")
    .sort_values("Verified Reports", ascending=False)
)

verified_language_summary["Percentage"] = (
    verified_language_summary["Verified Reports"]
    / len(verified_reports)
)

display(
    verified_language_summary.style.format({
        "Percentage": "{:.2%}"
    })
)

# Compare all reports with verified reports
language_comparison = language_summary.merge(
    verified_language_summary[
        ["Language", "Verified Reports"]
    ],
    on="Language",
    how="left"
)

language_comparison["Verified Reports"] = (
    language_comparison["Verified Reports"]
    .fillna(0)
    .astype(int)
)

language_comparison["Verified Coverage"] = (
    language_comparison["Verified Reports"]
    / language_comparison["Reports"]
)

display(
    language_comparison[
        [
            "Language",
            "LanguageName",
            "Reports",
            "Verified Reports",
            "Verified Coverage",
        ]
    ].style.format({
        "Verified Coverage": "{:.2%}"
    })
)

# Check for duplicate reports among the verified studies
verified_duplicates = (
    verified_reports["Report"]
    .duplicated(keep=False)
)

print(
    "Verified studies with duplicated report text:",
    verified_duplicates.sum()
)

# Show the verified label prevalence
verified_label_summary = pd.DataFrame({
    "Positive": verified_reports[LABELS].eq(1).sum(),
    "Negative": verified_reports[LABELS].eq(0).sum(),
})

verified_label_summary["Positive Rate"] = (
    verified_label_summary["Positive"]
    / len(verified_reports)
)

display(
    verified_label_summary.style.format({
        "Positive Rate": "{:.2%}"
    })
)

In [ ]:
# Check the GPU and load a multilingual instruction model

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. Open Notebook Settings and select a GPU accelerator."
    )

print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.eval()

print("Model loaded successfully.")
print("Model:", MODEL_NAME)

In [ ]:
import json
import re
import pandas as pd
import torch


# ---------------------------------------------------------
# Verify prerequisites
# ---------------------------------------------------------

required_variables = [
    "model",
    "tokenizer",
    "LABELS",
    "report_analysis",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the preceding notebook cells first. "
        "Missing variables: "
        + ", ".join(missing_variables)
    )


# Recreate the verified subset if necessary
verified_reports = report_analysis[
    report_analysis[LABELS].notna().all(axis=1)
].copy()

print("Verified reports:", len(verified_reports))


# ---------------------------------------------------------
# Complete prompt, including anatomical corrections
# ---------------------------------------------------------

SYSTEM_PROMPT = """
You are extracting 12 structured findings from a knee MRI
radiology report. The report may be written in any language.

Return exactly one JSON object with these keys:
"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
"Medial OA", "Lateral OA", "PF OA", "Effusion",
"Synovitis", "Baker's", "Contusion", "Fracture".

Allowed values:
1 = present
0 = absent, intact, normal, or explicitly negated
null = genuinely not evaluated, not mentioned, or uncertain

Use the FINDINGS and IMPRESSION.
Do not infer a current abnormality from the clinical history
or examination indication.

GENERAL NORMAL-FINDING RULES:

- "Cruciate ligaments normal/intact" means ACL = 0.
- "Collateral ligaments normal/intact" means MCL = 0.
- "Menisci normal" or "no meniscal tear" means both meniscus
  labels = 0.
- Normal femorotibial cartilage means Medial OA = 0 and
  Lateral OA = 0.
- Normal marrow, no marrow signal abnormality, or no osseous
  abnormality supports Contusion = 0 and Fracture = 0.

ACL AND MCL:

- Current sprain, partial tear, or complete tear = 1.
- Mucoid degeneration without injury or tear is not positive.
- Normal, intact, or no tear = 0.

MENISCI:

- Current tear or rupture = 1.
- Degeneration without a tear = 0.
- Prior surgery, truncation, or marginal amputation without a
  current tear is not automatically positive.

OSTEOARTHRITIS AND CARTILAGE LOSS:

The knee has three separate cartilage compartments.

1. Medial tibiofemoral compartment:
   This includes the medial femoral condyle and medial tibial
   plateau. Only abnormalities in this compartment affect
   "Medial OA".

2. Lateral tibiofemoral compartment:
   This includes the lateral femoral condyle and lateral tibial
   plateau. Only abnormalities in this compartment affect
   "Lateral OA".

3. Patellofemoral compartment:
   This includes the patella and the entire femoral trochlea.
   Any trochlear cartilage abnormality belongs to "PF OA".

The words "medial" and "lateral" do not automatically indicate
Medial OA or Lateral OA. The anatomical structure must also be
part of the tibiofemoral compartment.

Examples:

- A lesion of the medial aspect of the femoral trochlea affects
  PF OA, not Medial OA.
- A lesion of the lateral aspect of the femoral trochlea affects
  PF OA, not Lateral OA.
- Normal femorotibial cartilage means Medial OA = 0 and
  Lateral OA = 0.
- Grade 3, grade 4, or full-thickness trochlear cartilage damage
  means PF OA = 1 even if the patellar cartilage is normal.

For each relevant compartment:

- Osteoarthritis, advanced chondropathy, grade 3 or grade 4
  cartilage damage, full-thickness cartilage loss, or a
  full-thickness chondral defect = 1.
- Normal cartilage or explicit absence of degeneration = 0.

Example report statement:

"Normal medial and lateral femorotibial cartilage. Grade 4
full-thickness chondral defect of the medial trochlear surface.
Patellar cartilage is normal."

Correct OA output:

"Medial OA": 0
"Lateral OA": 0
"PF OA": 1

EFFUSION:

- Joint effusion, increased intra-articular fluid, or
  hydrarthrosis = 1.
- Physiologic or normal fluid, or no effusion = 0.

SYNOVITIS:

- Synovial inflammation, thickening, hypertrophy, or
  proliferation = 1.
- Effusion alone does not imply synovitis.
- If the synovium is not addressed, return null.

BAKER'S CYST:

- Baker cyst, popliteal cyst, or
  gastrocnemius-semimembranosus bursal cyst = 1.
- No Baker cyst or no pathological popliteal cyst = 0.

CONTUSION:

- Traumatic bone contusion or bone bruise = 1.
- Degenerative subchondral edema alone is not a contusion.
- Normal marrow or no bone bruise = 0.

FRACTURE:

- Current fracture = 1.
- No fracture or normal osseous structures = 0.

Respect negation and uncertainty.
Return JSON only, without Markdown or explanation.
"""


# ---------------------------------------------------------
# Response parser
# ---------------------------------------------------------

def parse_model_response(raw_response):
    """
    Parse one model response into the 12 required states.
    """

    json_match = re.search(
        r"\{.*\}",
        raw_response,
        flags=re.DOTALL,
    )

    if not json_match:
        raise ValueError(
            "The response did not contain a JSON object:\n"
            + raw_response
        )

    parsed_response = json.loads(
        json_match.group(0)
    )

    predictions = {}

    for label in LABELS:
        value = parsed_response.get(label, None)

        if value not in (0, 1, None):
            value = None

        predictions[label] = value

    return predictions


# ---------------------------------------------------------
# Single-report extraction function
# ---------------------------------------------------------

def extract_report_labels(report):
    """
    Extract the 12 competition states from one report.
    """

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                "Classify this knee MRI report:\n\n"
                + str(report)
            ),
        },
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=6000,
    ).to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[1]

    new_tokens = generated[
        0,
        input_length:
    ]

    raw_response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    predictions = parse_model_response(
        raw_response
    )

    return predictions, raw_response


# Remove irrelevant deterministic-generation warnings
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None


# ---------------------------------------------------------
# Test the complete prompt on the development report
# ---------------------------------------------------------

test_row = verified_reports.iloc[0]

predicted_labels, raw_response = extract_report_labels(
    test_row["Report"]
)

comparison = pd.DataFrame(
    {
        "Verified": [
            int(test_row[label])
            for label in LABELS
        ],
        "Model": [
            predicted_labels[label]
            for label in LABELS
        ],
    },
    index=LABELS,
)

comparison.index.name = "Label"

covered = comparison["Model"].notna()

print("\nStudy:", test_row["StudyInstanceUID"])
print("Language:", test_row["LanguageName"])

print("\nRaw model response:")
print(raw_response)

print(
    "\nCoverage:",
    f"{covered.sum()}/{len(LABELS)}",
    f"({covered.mean():.1%})",
)

if covered.any():
    covered_accuracy = (
        comparison.loc[
            covered,
            "Verified",
        ].astype(int)
        == comparison.loc[
            covered,
            "Model",
        ].astype(int)
    ).mean()

    print(
        "Accuracy on non-null predictions:",
        f"{covered_accuracy:.1%}",
    )

display(comparison)

In [ ]:
import json
import os
import time
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

CHECKPOINT_PATH = Path(
    "/kaggle/working/qwen_verified_evaluation.csv"
)

# ---------------------------------------------------------
# Atomic checkpoint: replace the previous file only after
# the new checkpoint has been fully written.
# ---------------------------------------------------------

def save_evaluation_checkpoint(frame):
    temporary_path = CHECKPOINT_PATH.with_suffix(".tmp")

    frame.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, CHECKPOINT_PATH)

DEVELOPMENT_STUDY_ID = test_row["StudyInstanceUID"]

evaluation_reports = verified_reports[
    verified_reports["StudyInstanceUID"]
    != DEVELOPMENT_STUDY_ID
].copy()

print("Prompt-development studies: 1")
print("Unseen evaluation studies:", len(evaluation_reports))
print("Checkpoint:", CHECKPOINT_PATH)


# ---------------------------------------------------------
# Resume an interrupted evaluation if possible
# ---------------------------------------------------------

if CHECKPOINT_PATH.exists():
    evaluation_results = pd.read_csv(CHECKPOINT_PATH)

    # The resume set must exclude rows that failed for any reason,
    # so the per-study loop re-attempts them on the next run.
    # The error row itself is still recorded (with ParsingError
    # set), so nothing is hidden; it is simply no longer treated
    # as "done".
    completed_studies = set(
        evaluation_results.loc[
            evaluation_results["ParsingError"].fillna("") == "",
            "StudyInstanceUID",
        ]
    )

    print(
        "Resuming from checkpoint:",
        len(completed_studies),
        "studies already completed.",
    )
else:
    evaluation_results = pd.DataFrame()
    completed_studies = set()


new_results = []


# ---------------------------------------------------------
# Evaluate each unseen verified report
# ---------------------------------------------------------

for _, row in tqdm(
    evaluation_reports.iterrows(),
    total=len(evaluation_reports),
    desc="Evaluating verified reports",
):
    study_id = str(row["StudyInstanceUID"])

    if study_id in completed_studies:
        continue

    result = {
        "StudyInstanceUID": study_id,
        "Language": row["Language"],
        "LanguageName": row["LanguageName"],
        "ParsingError": "",
        "RawResponse": "",
    }

    # Store verified labels
    for label in LABELS:
        result[f"true_{label}"] = int(row[label])

    try:
        predictions, raw_response = extract_report_labels(
            row["Report"]
        )

        result["RawResponse"] = raw_response

        for label in LABELS:
            result[f"pred_{label}"] = predictions[label]

    except Exception as error:
        result["ParsingError"] = str(error)

        for label in LABELS:
            result[f"pred_{label}"] = None

    new_results.append(result)

    # Save every five new studies
    if len(new_results) % 5 == 0:
        new_frame = pd.DataFrame(new_results)

        if evaluation_results.empty:
            saved_results = new_frame
        else:
            saved_results = pd.concat(
                [evaluation_results, new_frame],
                ignore_index=True,
            )

        save_evaluation_checkpoint(saved_results)

        print(
            f"Checkpoint saved: "
            f"{len(saved_results)}/{len(evaluation_reports)}"
        )


# ---------------------------------------------------------
# Save the final results
# ---------------------------------------------------------

if new_results:
    new_frame = pd.DataFrame(new_results)

    if evaluation_results.empty:
        evaluation_results = new_frame
    else:
        evaluation_results = pd.concat(
            [evaluation_results, new_frame],
            ignore_index=True,
        )

evaluation_results = (
    evaluation_results
    .drop_duplicates(
        subset="StudyInstanceUID",
        keep="last",
    )
    .reset_index(drop=True)
)

save_evaluation_checkpoint(evaluation_results)

print(
    "\nEvaluation complete:",
    f"{len(evaluation_results)}/{len(evaluation_reports)}"
)

print(
    "Parsing failures:",
    evaluation_results["ParsingError"]
    .fillna("")
    .ne("")
    .sum()
)


# ---------------------------------------------------------
# Convert prediction columns to numeric values
# ---------------------------------------------------------

true_columns = [
    f"true_{label}"
    for label in LABELS
]

prediction_columns = [
    f"pred_{label}"
    for label in LABELS
]

for column in true_columns + prediction_columns:
    evaluation_results[column] = pd.to_numeric(
        evaluation_results[column],
        errors="coerce",
    )


# ---------------------------------------------------------
# Calculate overall metrics
# ---------------------------------------------------------

all_true = evaluation_results[
    true_columns
].to_numpy(dtype=float)

all_predictions = evaluation_results[
    prediction_columns
].to_numpy(dtype=float)

covered_mask = ~np.isnan(all_predictions)

overall_coverage = covered_mask.mean()

if covered_mask.any():
    overall_accuracy = (
        all_predictions[covered_mask]
        == all_true[covered_mask]
    ).mean()
else:
    overall_accuracy = np.nan

print("\nOverall unseen-report results:")
print(f"Coverage: {overall_coverage:.2%}")
print(f"Accuracy on covered labels: {overall_accuracy:.2%}")


# ---------------------------------------------------------
# Calculate metrics for each label
# ---------------------------------------------------------

label_metrics = []

for label in LABELS:
    true_values = evaluation_results[
        f"true_{label}"
    ]

    predicted_values = evaluation_results[
        f"pred_{label}"
    ]

    covered = predicted_values.notna()

    y_true = true_values[covered].astype(int)
    y_pred = predicted_values[covered].astype(int)

    if len(y_true) > 0:
        accuracy = (y_true == y_pred).mean()

        true_positive = (
            (y_true == 1) & (y_pred == 1)
        ).sum()

        true_negative = (
            (y_true == 0) & (y_pred == 0)
        ).sum()

        false_positive = (
            (y_true == 0) & (y_pred == 1)
        ).sum()

        false_negative = (
            (y_true == 1) & (y_pred == 0)
        ).sum()

        sensitivity_denominator = (
            true_positive + false_negative
        )

        specificity_denominator = (
            true_negative + false_positive
        )

        sensitivity = (
            true_positive / sensitivity_denominator
            if sensitivity_denominator > 0
            else np.nan
        )

        specificity = (
            true_negative / specificity_denominator
            if specificity_denominator > 0
            else np.nan
        )
    else:
        accuracy = np.nan
        true_positive = 0
        true_negative = 0
        false_positive = 0
        false_negative = 0
        sensitivity = np.nan
        specificity = np.nan

    label_metrics.append({
        "Label": label,
        "Coverage": covered.mean(),
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "TP": true_positive,
        "TN": true_negative,
        "FP": false_positive,
        "FN": false_negative,
    })


label_metrics = pd.DataFrame(
    label_metrics
).set_index("Label")

print("\nResults by label:")

display(
    label_metrics.style.format({
        "Coverage": "{:.2%}",
        "Accuracy": "{:.2%}",
        "Sensitivity": "{:.2%}",
        "Specificity": "{:.2%}",
    })
)


# ---------------------------------------------------------
# Calculate metrics for each language
# ---------------------------------------------------------

language_metrics = []

for (
    language_code,
    language_name
), group in evaluation_results.groupby(
    ["Language", "LanguageName"]
):
    language_true = group[
        true_columns
    ].to_numpy(dtype=float)

    language_predictions = group[
        prediction_columns
    ].to_numpy(dtype=float)

    language_covered = ~np.isnan(
        language_predictions
    )

    coverage = language_covered.mean()

    if language_covered.any():
        accuracy = (
            language_predictions[language_covered]
            == language_true[language_covered]
        ).mean()
    else:
        accuracy = np.nan

    language_metrics.append({
        "Language": language_code,
        "LanguageName": language_name,
        "Reports": len(group),
        "Coverage": coverage,
        "Accuracy": accuracy,
    })


language_metrics = (
    pd.DataFrame(language_metrics)
    .sort_values(
        ["Reports", "Accuracy"],
        ascending=[False, False],
    )
)

print("\nResults by language:")

display(
    language_metrics.style.format({
        "Coverage": "{:.2%}",
        "Accuracy": "{:.2%}",
    })
)

print("\nSaved evaluation file:")
print(CHECKPOINT_PATH)

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

CALIBRATION_PATH = Path(
    "/kaggle/working/qwen_state_calibration.csv"
)

# Higher values produce more conservative probabilities.
# This shrinks estimates from small groups toward prevalence.
PRIOR_STRENGTH = 8.0


# ---------------------------------------------------------
# Calculate empirical-Bayes calibration
# ---------------------------------------------------------

calibration_rows = []

for label in LABELS:
    true_column = f"true_{label}"
    prediction_column = f"pred_{label}"

    true_values = pd.to_numeric(
        evaluation_results[true_column],
        errors="coerce",
    )

    predicted_values = pd.to_numeric(
        evaluation_results[prediction_column],
        errors="coerce",
    )

    base_rate = true_values.mean()

    # Convert missing model outputs into a distinct "null" state
    states = predicted_values.apply(
        lambda value: (
            "null"
            if pd.isna(value)
            else str(int(value))
        )
    )

    for state in ["0", "1", "null"]:
        state_mask = states == state

        count = int(state_mask.sum())

        if count == 0:
            continue

        state_true_values = true_values[state_mask]

        positives = int(
            state_true_values.eq(1).sum()
        )

        negatives = int(
            state_true_values.eq(0).sum()
        )

        raw_positive_rate = (
            positives / count
        )

        # Shrink the observed state rate toward the overall
        # verified prevalence for this label
        soft_probability = (
            positives
            + PRIOR_STRENGTH * base_rate
        ) / (
            count + PRIOR_STRENGTH
        )

        sample_confidence = (
            count
            / (count + PRIOR_STRENGTH)
        )

        # Measure how informative this state is compared with
        # simply predicting the label's base prevalence
        if soft_probability >= base_rate:
            maximum_distance = 1.0 - base_rate
        else:
            maximum_distance = base_rate

        if maximum_distance > 0:
            information_strength = (
                abs(soft_probability - base_rate)
                / maximum_distance
            )
        else:
            information_strength = 0.0

        weak_weight = (
            sample_confidence
            * information_strength
        )

        # Accuracy only has a natural interpretation for
        # hard states 0 and 1
        if state == "1":
            hard_state_accuracy = raw_positive_rate
        elif state == "0":
            hard_state_accuracy = (
                1.0 - raw_positive_rate
            )
        else:
            hard_state_accuracy = np.nan

        calibration_rows.append({
            "Label": label,
            "State": state,
            "Count": count,
            "Positives": positives,
            "Negatives": negatives,
            "BaseRate": base_rate,
            "RawPositiveRate": raw_positive_rate,
            "SoftProbability": soft_probability,
            "HardStateAccuracy": hard_state_accuracy,
            "SampleConfidence": sample_confidence,
            "InformationStrength": information_strength,
            "WeakWeight": weak_weight,
        })


calibration_table = pd.DataFrame(
    calibration_rows
)

calibration_table.to_csv(
    CALIBRATION_PATH,
    index=False,
)


# ---------------------------------------------------------
# Display the calibrated mapping
# ---------------------------------------------------------

print("Calibrated report-model states:")

display(
    calibration_table[
        [
            "Label",
            "State",
            "Count",
            "Positives",
            "Negatives",
            "BaseRate",
            "RawPositiveRate",
            "SoftProbability",
            "HardStateAccuracy",
            "WeakWeight",
        ]
    ].style.format({
        "BaseRate": "{:.2%}",
        "RawPositiveRate": "{:.2%}",
        "SoftProbability": "{:.3f}",
        "HardStateAccuracy": "{:.2%}",
        "WeakWeight": "{:.3f}",
    })
)


# ---------------------------------------------------------
# Show the most and least informative states
# ---------------------------------------------------------

print("\nMost informative model states:")

display(
    calibration_table[
        calibration_table["State"] != "null"
    ]
    .sort_values(
        "WeakWeight",
        ascending=False,
    )
    .head(15)[
        [
            "Label",
            "State",
            "Count",
            "SoftProbability",
            "HardStateAccuracy",
            "WeakWeight",
        ]
    ]
    .style.format({
        "SoftProbability": "{:.3f}",
        "HardStateAccuracy": "{:.2%}",
        "WeakWeight": "{:.3f}",
    })
)

print("\nLeast informative model states:")

display(
    calibration_table[
        calibration_table["State"] != "null"
    ]
    .sort_values(
        "WeakWeight",
        ascending=True,
    )
    .head(15)[
        [
            "Label",
            "State",
            "Count",
            "SoftProbability",
            "HardStateAccuracy",
            "WeakWeight",
        ]
    ]
    .style.format({
        "SoftProbability": "{:.3f}",
        "HardStateAccuracy": "{:.2%}",
        "WeakWeight": "{:.3f}",
    })
)

print("\nCalibration saved to:")
print(CALIBRATION_PATH)

In [ ]:
import json
import re
import time
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm


# ---------------------------------------------------------
# Configure tokenizer for batched decoder-only generation
# ---------------------------------------------------------

tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def parse_model_response(raw_response):
    """
    Convert one model response into the 12 report states.
    """

    json_match = re.search(
        r"\{.*\}",
        raw_response,
        flags=re.DOTALL,
    )

    if not json_match:
        raise ValueError(
            "No JSON object found in response."
        )

    parsed = json.loads(
        json_match.group(0)
    )

    predictions = {}

    for label in LABELS:
        value = parsed.get(label, None)

        if value not in (0, 1, None):
            value = None

        predictions[label] = value

    return predictions


def extract_report_labels_batch(reports):
    """
    Extract report states for several reports simultaneously.
    """

    prompts = []

    for report in reports:
        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    "Classify this knee MRI report:\n\n"
                    + str(report)
                ),
            },
        ]

        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        prompts.append(formatted_prompt)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=6000,
    ).to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Generated tokens begin after the padded input width
    input_width = inputs["input_ids"].shape[1]

    batch_results = []

    for sequence in generated:
        new_tokens = sequence[input_width:]

        raw_response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        try:
            predictions = parse_model_response(
                raw_response
            )

            parsing_error = ""

        except Exception as error:
            predictions = {
                label: None
                for label in LABELS
            }

            parsing_error = str(error)

        batch_results.append({
            "Predictions": predictions,
            "RawResponse": raw_response,
            "ParsingError": parsing_error,
        })

    return batch_results


# ---------------------------------------------------------
# Select a multilingual preflight sample
# Up to two verified studies from every language
# ---------------------------------------------------------

batch_test_rows = (
    evaluation_reports
    .sort_values(
        ["Language", "StudyInstanceUID"]
    )
    .groupby(
        "Language",
        group_keys=False,
    )
    .head(2)
    .reset_index(drop=True)
)

print(
    "Preflight reports:",
    len(batch_test_rows)
)

display(
    batch_test_rows[
        [
            "StudyInstanceUID",
            "LanguageName",
        ]
    ]
)


# ---------------------------------------------------------
# Run in batches of four
# ---------------------------------------------------------

BATCH_SIZE = 4

batch_test_results = []

start_time = time.time()

for start_index in tqdm(
    range(0, len(batch_test_rows), BATCH_SIZE),
    desc="Batch preflight",
):
    batch_rows = batch_test_rows.iloc[
        start_index:
        start_index + BATCH_SIZE
    ]

    outputs = extract_report_labels_batch(
        batch_rows["Report"].tolist()
    )

    for (_, row), output in zip(
        batch_rows.iterrows(),
        outputs,
    ):
        result = {
            "StudyInstanceUID": str(
                row["StudyInstanceUID"]
            ),
            "Language": row["Language"],
            "LanguageName": row["LanguageName"],
            "ParsingError": output["ParsingError"],
            "RawResponse": output["RawResponse"],
        }

        for label in LABELS:
            result[f"batch_{label}"] = (
                output["Predictions"][label]
            )

        batch_test_results.append(result)


elapsed_seconds = time.time() - start_time

batch_test_results = pd.DataFrame(
    batch_test_results
)

print("\nBatch preflight complete.")
print(
    "Elapsed time:",
    f"{elapsed_seconds:.1f} seconds"
)
print(
    "Average per report:",
    f"{elapsed_seconds / len(batch_test_rows):.2f} seconds"
)
print(
    "Estimated time for 4,407 reports:",
    f"{elapsed_seconds / len(batch_test_rows) * 4407 / 3600:.2f} hours"
)

print(
    "Parsing failures:",
    batch_test_results["ParsingError"]
    .fillna("")
    .ne("")
    .sum()
)


# ---------------------------------------------------------
# Compare batched states with previous sequential states
# ---------------------------------------------------------

sequential_columns = [
    "StudyInstanceUID"
] + [
    f"pred_{label}"
    for label in LABELS
]

comparison_data = batch_test_results.merge(
    evaluation_results[sequential_columns],
    on="StudyInstanceUID",
    how="left",
)

agreement_rows = []

for label in LABELS:
    sequential_values = pd.to_numeric(
        comparison_data[f"pred_{label}"],
        errors="coerce",
    )

    batch_values = pd.to_numeric(
        comparison_data[f"batch_{label}"],
        errors="coerce",
    )

    matching = (
        sequential_values.eq(batch_values)
        | (
            sequential_values.isna()
            & batch_values.isna()
        )
    )

    agreement_rows.append({
        "Label": label,
        "Agreement": matching.mean(),
        "Differences": int((~matching).sum()),
    })


batch_agreement = pd.DataFrame(
    agreement_rows
).set_index("Label")

all_agreements = (
    batch_agreement["Agreement"].mean()
)

print(
    "\nAverage batch/sequential agreement:",
    f"{all_agreements:.2%}"
)

display(
    batch_agreement.style.format({
        "Agreement": "{:.2%}"
    })
)


# ---------------------------------------------------------
# Preview the batch outputs
# ---------------------------------------------------------

preview_columns = [
    "StudyInstanceUID",
    "LanguageName",
    "ParsingError",
] + [
    f"batch_{label}"
    for label in LABELS
]

display(
    batch_test_results[
        preview_columns
    ]
)

In [ ]:
import gc
import os
import time
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm.auto import tqdm


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

STATE_PATH = Path(
    "/kaggle/working/qwen_all_report_states.csv"
)

BATCH_SIZE = 4
SAVE_EVERY_REPORTS = 100


# ---------------------------------------------------------
# Prepare all reports
# ---------------------------------------------------------

all_reports = report_analysis[
    [
        "StudyInstanceUID",
        "Language",
        "LanguageName",
        "Report",
    ]
].copy()

all_reports["StudyInstanceUID"] = (
    all_reports["StudyInstanceUID"].astype(str)
)

print("Total reports:", len(all_reports))
print("Output file:", STATE_PATH)


# ---------------------------------------------------------
# Load previous progress
# ---------------------------------------------------------

if STATE_PATH.exists():
    saved_results = pd.read_csv(
        STATE_PATH,
        dtype={
            "StudyInstanceUID": str,
        },
    )

    saved_results = (
        saved_results
        .drop_duplicates(
            subset="StudyInstanceUID",
            keep="last",
        )
        .reset_index(drop=True)
    )

    completed_ids = set(
        saved_results["StudyInstanceUID"]
    )

    print(
        "Resuming:",
        len(completed_ids),
        "reports already completed.",
    )

else:
    saved_results = pd.DataFrame()
    completed_ids = set()

    print("Starting a new full-report run.")


pending_reports = all_reports[
    ~all_reports["StudyInstanceUID"].isin(
        completed_ids
    )
].reset_index(drop=True)

print("Reports remaining:", len(pending_reports))


# ---------------------------------------------------------
# Safe batched inference with automatic fallback
# ---------------------------------------------------------

def safely_extract_batch(reports):
    """
    Run batched extraction. If a batch causes a GPU error,
    recursively divide it into smaller batches.
    """

    try:
        return extract_report_labels_batch(
            reports
        )

    except RuntimeError as error:
        print(
            "\nBatch runtime error; retrying with "
            "smaller batches:"
        )
        print(str(error)[:500])

        gc.collect()
        torch.cuda.empty_cache()

        if len(reports) == 1:
            return [{
                "Predictions": {
                    label: None
                    for label in LABELS
                },
                "RawResponse": "",
                "ParsingError": str(error),
            }]

        midpoint = len(reports) // 2

        first_half = safely_extract_batch(
            reports[:midpoint]
        )

        second_half = safely_extract_batch(
            reports[midpoint:]
        )

        return first_half + second_half


# ---------------------------------------------------------
# Checkpoint-saving function
# ---------------------------------------------------------

def save_checkpoint(current_results, buffer):
    """
    Add buffered records to the saved table and write CSV.
    """

    if buffer:
        buffer_frame = pd.DataFrame(buffer)

        if current_results.empty:
            current_results = buffer_frame
        else:
            current_results = pd.concat(
                [
                    current_results,
                    buffer_frame,
                ],
                ignore_index=True,
            )

    current_results = (
        current_results
        .drop_duplicates(
            subset="StudyInstanceUID",
            keep="last",
        )
        .reset_index(drop=True)
    )

    temporary_path = STATE_PATH.with_suffix(".tmp")

    current_results.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, STATE_PATH)

    return current_results


# ---------------------------------------------------------
# Process every remaining report
# ---------------------------------------------------------

buffer = []
reports_since_save = 0

start_time = time.time()

batch_starts = range(
    0,
    len(pending_reports),
    BATCH_SIZE,
)

progress_bar = tqdm(
    batch_starts,
    total=int(
        np.ceil(
            len(pending_reports)
            / BATCH_SIZE
        )
    ),
    desc="Extracting all report states",
)


try:
    for start_index in progress_bar:
        batch_rows = pending_reports.iloc[
            start_index:
            start_index + BATCH_SIZE
        ]

        batch_outputs = safely_extract_batch(
            batch_rows["Report"].tolist()
        )

        for (_, row), output in zip(
            batch_rows.iterrows(),
            batch_outputs,
        ):
            record = {
                "StudyInstanceUID": str(
                    row["StudyInstanceUID"]
                ),
                "Language": row["Language"],
                "LanguageName": row["LanguageName"],
                "ParsingError": output[
                    "ParsingError"
                ],
                "RawResponse": output[
                    "RawResponse"
                ],
            }

            for label in LABELS:
                record[f"state_{label}"] = (
                    output["Predictions"][label]
                )

            buffer.append(record)
            reports_since_save += 1

        # Save approximately every 100 reports
        if reports_since_save >= SAVE_EVERY_REPORTS:
            saved_results = save_checkpoint(
                saved_results,
                buffer,
            )

            buffer = []
            reports_since_save = 0

            completed_total = len(saved_results)

            elapsed = time.time() - start_time

            newly_completed = max(
                completed_total - len(completed_ids),
                1,
            )

            seconds_per_report = (
                elapsed / newly_completed
            )

            reports_left = (
                len(all_reports)
                - completed_total
            )

            estimated_hours_left = (
                reports_left
                * seconds_per_report
                / 3600
            )

            progress_bar.set_postfix({
                "saved": completed_total,
                "hours_left": (
                    f"{estimated_hours_left:.2f}"
                ),
            })

finally:
    # Saves on normal completion or a catchable interruption.
    # A forced session shutdown can still lose draft-session files.
    saved_results = save_checkpoint(
        saved_results,
        buffer,
    )

    buffer = []

    progress_bar.close()

# ---------------------------------------------------------
# Final save
# ---------------------------------------------------------

saved_results = save_checkpoint(
    saved_results,
    buffer,
)

elapsed = time.time() - start_time

print("\nFull report extraction finished.")
print("Saved reports:", len(saved_results))
print(
    "Time used in this run:",
    f"{elapsed / 3600:.2f} hours",
)


# ---------------------------------------------------------
# Validate the output
# ---------------------------------------------------------

expected_ids = set(
    all_reports["StudyInstanceUID"]
)

actual_ids = set(
    saved_results["StudyInstanceUID"]
)

missing_ids = expected_ids - actual_ids
unexpected_ids = actual_ids - expected_ids

parsing_failures = (
    saved_results["ParsingError"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print("\nValidation:")
print("Expected reports:", len(expected_ids))
print("Saved reports:", len(actual_ids))
print("Missing reports:", len(missing_ids))
print("Unexpected reports:", len(unexpected_ids))
print("Parsing failures:", parsing_failures)
print(
    "Duplicate study IDs:",
    saved_results["StudyInstanceUID"]
    .duplicated()
    .sum(),
)


if missing_ids:
    print(
        "\nFirst missing study IDs:",
        list(missing_ids)[:10],
    )


# ---------------------------------------------------------
# Show state distributions
# ---------------------------------------------------------

state_distribution_rows = []

for label in LABELS:
    state_column = f"state_{label}"

    numeric_states = pd.to_numeric(
        saved_results[state_column],
        errors="coerce",
    )

    state_distribution_rows.append({
        "Label": label,
        "State 0": int(
            numeric_states.eq(0).sum()
        ),
        "State 1": int(
            numeric_states.eq(1).sum()
        ),
        "Null": int(
            numeric_states.isna().sum()
        ),
        "Positive State Rate": (
            numeric_states.eq(1).mean()
        ),
        "Coverage": (
            numeric_states.notna().mean()
        ),
    })


state_distribution = pd.DataFrame(
    state_distribution_rows
).set_index("Label")

print("\nFull-dataset state distribution:")

display(
    state_distribution.style.format({
        "Positive State Rate": "{:.2%}",
        "Coverage": "{:.2%}",
    })
)

print("\nReport states saved to:")
print(STATE_PATH)